# Container Build & Run Quickstart

This notebook demonstrates how to build and validate a benchmark container for a single repository commit using the refactored Datasmith orchestration APIs.

## Usage
1. Update the repository parameters below.
2. Point `CONTEXT_REGISTRY_PATH` at a registry JSON that includes your repo/sha (or rely on the default context).
3. Run the notebook top-to-bottom.

> **Note:** Docker, `asv`, and the Datasmith Python package must be available in the current environment.

In [1]:
%load_ext autoreload
%autoreload 2
%cd /mnt/sdd1/atharvas/formulacode/datasmith
import json
from pathlib import Path

import pandas as pd

/mnt/sdd1/atharvas/formulacode/datasmith


In [2]:
results_df = pd.DataFrame([
    json.loads(line)
    for line in Path("scratch/artifacts/pipeflush/symbolic_synthesis/results.jsonl").read_text().splitlines()
    if line != "null"
])
errors = results_df.query("not can_install")
print(errors.shape)
offenders = set(errors["dry_run_log"].str.extract(r"Because ([\w\-]*) was not found").dropna().values.flatten())
offenders.update(
    set(errors["dry_run_log"].str.extract(r"Because there are no versions of ([\w\-]*)").dropna().values.flatten())
)
# Because you require pyvisa==1.11.1 and pyvisa>=1.11.3,<1.12.dev0, we can
offenders

(9281, 15)


{'0-29-21',
 '0-29-30',
 '0-29-32',
 '0-29-33',
 '1-0',
 '1-11-2',
 '1-12',
 '1-14-0',
 '1-2',
 '1-22-0',
 '1-23-5',
 '1-8-1',
 '1-9-1',
 '2-18-4',
 '2-2',
 '3-0',
 '3-0-0a10',
 '3-0-0a11',
 '3-0-5',
 '3-1-2',
 '3-2-0',
 '59-2-0',
 'backports',
 'c-distances-openmp',
 'cartopy-userconfig',
 'cdms2',
 'column-parsers',
 'copy-reg',
 'cpickle',
 'cprofile',
 'cryptodome',
 'cupyx',
 'flatted',
 'givens-elimination',
 'h5r',
 'h5s',
 'h5z',
 'imp',
 'libreader',
 'mo-pack',
 'mpl-toolkits',
 'nattype',
 'omniscidbe',
 'openjpeg',
 'patoolib',
 'peerplaysbase',
 'pnetdicom',
 'probabilistic-direction-getter',
 'pyhdk',
 'pymake',
 'pypocketfft',
 'pyqt',
 'pyqt4',
 'sksparse',
 'splitting',
 'stringio',
 'uninstall',
 'urlparse',
 'vectorized',
 'voyager-ext'}

In [14]:
import json
from pathlib import Path

import pandas as pd

from datasmith.core.models.task import Task
from datasmith.execution.resolution import analyze_commit

results_df = pd.DataFrame([
    json.loads(line)
    for line in Path("scratch/artifacts/pipeflush/symbolic_synthesis/results.jsonl").read_text().splitlines()
    if line != "null"
])
not_errors = results_df.query("can_install")
errors = results_df.query("not can_install")

d = errors.groupby("dry_run_log").head(1).to_dict(orient="records")

tasks = [
    Task(
        owner=row["repo_name"].split("/")[0],
        repo=row["repo_name"].split("/")[1],
        sha=row["sha"],
    )
    for row in not_errors.to_dict(orient="records")
]
print(len(tasks))
# for t in random.sample(tasks, 2):
#     task_requirements = analyze_commit(
#     sha=t.sha,
#     repo_name=f"{t.owner}/{t.repo}", bypass_cache=True)
#     if task_requirements['can_install']:
#         continue
#     print(t)
#     print(task_requirements['dry_run_log'])

3519


In [15]:
# CONTEXT_REGISTRY_PATH = Path('scratch/artifacts/context_registry_init.json')
OUTPUT_DIR = Path("scratch/notebooks/output").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")

Output directory: /mnt/sdd1/atharvas/formulacode/datasmith/scratch/notebooks/output


In [47]:
task = tasks[5]
print(task)
task_analysis = analyze_commit(sha=task.sha, repo_name=f"{task.owner}/{task.repo}", bypass_cache=True)
assert task_analysis and task_analysis["can_install"], "Task cannot be installed"
print(task_analysis)

Task(owner='devitocodes', repo='devito', sha='0d6197ba9cd9e5da388355216bb46e24e1251ee7', commit_date=0.0, env_payload='', python_version='', tag='pkg')
{'sha': '0d6197ba9cd9e5da388355216bb46e24e1251ee7', 'repo_name': 'devitocodes/devito', 'package_name': 'devito', 'package_version': '4.8.20.dev211+g0d6197ba9', 'python_version': '3.12', 'build_command': ['DEVITO_BENCHMARKS=1 python setup.py build && PIP_NO_BUILD_ISOLATION=false python -m pip wheel --no-deps --no-index -w {build_cache_dir} {build_dir}'], 'install_command': ["in-dir={env_dir} python -mpip install '{wheel_file}[tests]'"], 'final_dependencies': ['aiohappyeyeballs==2.6.1', 'aiohttp==3.12.15', 'aiosignal==1.4.0', 'anyio==4.10.0', 'anytree==2.13.0', 'argon2-cffi==25.1.0', 'argon2-cffi-bindings==25.1.0', 'arrow==1.3.0', 'asttokens==3.0.0', 'async-lru==2.0.5', 'attrs==25.3.0', 'babel==2.17.0', 'beautifulsoup4==4.13.4', 'bleach==6.2.0', 'bokeh==3.7.3', 'certifi==2025.8.3', 'cffi==1.17.1', 'cgen==2025.1', 'charset-normalizer==3.4.

In [48]:
from datasmith.notebooks.utils import merge_registries

registries = Path("scratch/").rglob("**/*context_registry*.json")
merged_json = merge_registries(list(registries))

In [49]:
import json

from datasmith.docker.context import ContextRegistry
from datasmith.notebooks.utils import update_cr

registry = update_cr(ContextRegistry.deserialize(payload=json.dumps(merged_json)))

In [ ]:
from datasmith.docker.context import DockerContext

TASK = Task(
    owner=task.owner,
    repo=task.repo,
    sha=task.sha,
    python_version=task_analysis["python_version"],
    env_payload=json.dumps({
        "dependencies": task_analysis["final_dependencies"],
    }),
)
pkg_task = TASK.with_tag("pkg")


context = registry.get(pkg_task)
context.building_data == DockerContext().building_data

False

In [51]:
# --- Connect to Docker ---
from datasmith.docker.orchestrator import get_docker_client

client = get_docker_client()
client

In [52]:
# --- Build the package image ---
from datasmith.docker.orchestrator import build_repo_sha_image

build_result = build_repo_sha_image(
    client=client,
    docker_ctx=context,
    task=pkg_task.with_tag("run"),
    run_id="notebook-run",
    force=True,
)
build_result

17:15:38 INFO     datasmith.docker.context: Force rebuild requested. Removing 'devitocodes-devito-0d6197ba9cd9e5da388355216bb46e24e1251ee7:run'.


17:15:39 INFO     datasmith.docker.context: $ docker build -t devitocodes-devito-0d6197ba9cd9e5da388355216bb46e24e1251ee7:run . --build-arg REPO_URL='https://www.github.com/devitocodes/devito' --build-arg COMMIT_SHA='0d6197ba9cd9e5da388355216bb46e24e1251ee7' --build-arg ENV_PAYLOAD='{"dependencies": ["aiohappyeyeballs==2.6.1", "aiohttp==3.12.15", "aiosignal==1.4.0", "anyio==4.10.0", "anytree==2.13.0", "argon2-cffi==25.1.0", "argon2-cffi-bindings==25.1.0", "arrow==1.3.0", "asttokens==3.0.0", "async-lru==2.0.5", "attrs==25.3.0", "babel==2.17.0", "beautifulsoup4==4.13.4", "bleach==6.2.0", "bokeh==3.7.3", "certifi==2025.8.3", "cffi==1.17.1", "cgen==2025.1", "charset-normalizer==3.4.3", "click==8.2.1", "cloudpickle==3.1.1", "codepy==2023.1", "comm==0.2.3", "contexttimer==0.3.3", "contourpy==1.3.3", "coverage==7.10.3", "cupy-cuda12x==13.5.1", "cycler==0.12.1", "dask==2025.7.0", "dask-cuda==25.8.0", "dask-labextension==7.0.0", "debugpy==1.8.16", "decorator==5.2.1", "defusedxml==0.7.1", "distr

BuildResult(ok=True, image_name='devitocodes-devito-0d6197ba9cd9e5da388355216bb46e24e1251ee7:run', image_id='sha256:1f783b95af9e569c5636ffbcd115ed663ac8f61212a8bae492091c5c7602fd27', rc=0, duration_s=156.6749622821808, stderr_tail='', stdout_tail='      ********************************************************************************\n\x1b\x1b\n\x1b\x1b  !!\n\x1b\x1b    class PyTest(orig.test):\n\x1b\x1b  /opt/conda/envs/asv_3.12/lib/python3.12/site-packages/setuptools/config/_apply_pyprojecttoml.py:61: SetuptoolsDeprecationWarning: License classifiers are deprecated.\n\x1b\x1b  !!\n\x1b\x1b\n\x1b\x1b          ********************************************************************************\n\x1b\x1b          Please consider removing the following classifiers in favor of a SPDX license expression:\n\x1b\x1b\n\x1b\x1b          License :: OSI Approved :: MIT License\n\x1b\x1b\n\x1b\x1b          See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.\n

In [53]:
# --- Run a quick validation profile ---
import asv

from datasmith.docker.validation import DockerValidator, ValidationConfig

raw_defaults = asv.machine.Machine.get_defaults()  # type: ignore[attr-defined]
machine_defaults = {k: str(v).replace(" ", "_") for k, v in raw_defaults.items()}

config = ValidationConfig(output_dir=OUTPUT_DIR, build_timeout=1800, run_timeout=600, tail_chars=4000)
validator = DockerValidator(
    client=client,
    context_registry=registry,
    machine_defaults=machine_defaults,
    config=config,
)

run_result = validator.validate_task(TASK.with_tag("run"), run_labels={})
run_result

17:18:18 INFO     datasmith.docker.validation: validate_acceptance: validating image 'devitocodes-devito-0d6197ba9cd9e5da388355216bb46e24e1251ee7:run'


AcceptanceResult(profile=ProfileValidationResult(ok=True, output='[profile] Running ASV baseline on 0d6197ba9cd9e5da388355216bb46e24e1251ee7\n· Discovering benchmarks\n· Running 8 total benchmarks (1 commits * 1 environments * 8 benchmarks)\n[ 0.00%] · For devito commit 0d6197ba <main>:\n[ 0.00%] ·· Building for existing-py_opt_conda_envs_asv_3.12_bin_python\n[ 0.00%] ·· Benchmarking existing-py_opt_conda_envs_asv_3.12_bin_python\n[ 6.25%] ··· Running (acoustic.IsotropicAcoustic.time_forward--)', duration_s=45.81995248794556, error=None), tests=TestValidationResult(ok=True, output="+ cd /workspace/repo\n+ set +ux\n+ '[' 0 -gt 0 ']'\n+ '[' -n 0d6197ba9cd9e5da388355216bb46e24e1251ee7 ']'\n+ FORMULACODE_BASE_COMMIT=0d6197ba9cd9e5da388355216bb46e24e1251ee7\n+ reset_repo_state 0d6197ba9cd9e5da388355216bb46e24e1251ee7\n+ local COMMIT_SHA=0d6197ba9cd9e5da388355216bb46e24e1251ee7\n++ git remote -v\n++ grep '(fetch)'\n++ awk '{print $2}'\n+ URL=https://www.github.com/devitocodes/devito\n+ [[ ht

In [ ]:
# --- Optional cleanup ---
from datasmith.docker.orchestrator import generate_run_labels

labels = generate_run_labels(TASK, run_id="notebook-run")
print("Labels for this run:", labels)
print("Use docker CLI to prune images/containers if desired.")